# Логистическая регрессия с кросс-валидацией

Модель использует 131 признак и L2-регуляризацию. Оцениваем прогнозы на пяти внешних фолдах. Порог каждого фолда подбираем по F1 на его обучающей части. Для заблокированных объявлений всегда ставим `is_deal = 0`.

## 1. Загрузить данные

Используем `features.csv` из `feature_engineering.ipynb`. Метки и признаки сопоставляем по `listing_id`.

In [13]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

In [14]:
FEATURES_PATH = Path("data_by_dewallen/features.csv")
VERIFIED_PATH = Path("data/verified.csv")
LISTINGS_PATH = Path("data/listings.csv")
OUTPUT_PATH = Path("submission.csv")

CV_SEEDS = [11, 23, 42, 71, 101, 13, 29, 59, 83, 107]
THRESHOLDS = np.round(np.arange(0.20, 0.71, 0.01), 2)
INNER_SEED = 42

In [15]:
features = pd.read_csv(FEATURES_PATH, index_col="listing_id")
verified = pd.read_csv(VERIFIED_PATH, usecols=["listing_id", "is_deal"], index_col="listing_id")
listings = pd.read_csv(LISTINGS_PATH, usecols=["listing_id", "close_type"]).drop_duplicates()
listings = listings.set_index("listing_id")

assert listings.index.is_unique and features.index.is_unique
assert set(features.index) == set(listings.index), "Пересчитайте признаки для всех объявлений"
assert verified.index.is_unique, "Повторяются ID в проверенной разметке"
assert verified.index.isin(features.index).all()
assert verified["is_deal"].isin([0, 1]).all(), "Некорректные метки в verified.csv"
assert features.columns.is_unique

feature_columns = features.columns.difference(["seller_id", "is_deal"], sort=False)
X = features[feature_columns]
y = verified["is_deal"]
X_train = X.loc[y.index]

print(f"Объявлений в матрице: {len(X)}")
print(f"Признаков: {X.shape[1]}")
print(f"Размеченных объявлений с признаками: {len(y)}")

Объявлений в матрице: 15274
Признаков: 131
Размеченных объявлений с признаками: 300


## 2. Создать модель

В числовом блоке заменяем бесконечности пропусками, заполняем пропуски медианой и стандартизируем значения. В категориальном блоке заполняем пропуски самым частым значением и применяем one-hot. Каждый фолд получает свои параметры подготовки, рассчитанные на его обучающей части.

Логистическая регрессия использует L2-регуляризацию с `C=1`.

In [16]:
def clean_numbers(frame):
    return frame.astype(float).replace([np.inf, -np.inf], np.nan)


def clean_categories(frame):
    strings = frame.astype("string")
    return strings.astype(object).where(strings.notna(), np.nan)


numeric_pipeline = make_pipeline(
    FunctionTransformer(clean_numbers, feature_names_out="one-to-one"),
    SimpleImputer(strategy="median"),
    StandardScaler(),
)
categorical_pipeline = make_pipeline(
    FunctionTransformer(clean_categories, feature_names_out="one-to-one"),
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore"),
)
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, make_column_selector(dtype_include=["number", "bool"])),
    ("categorical", categorical_pipeline, make_column_selector(dtype_exclude=["number", "bool"])),
])
model = make_pipeline(
    preprocessor,
    LogisticRegression(C=1.0, max_iter=2000),
)

## 3. Оценить качество на пяти фолдах

Внутри обучающей части каждого внешнего фолда строим прогнозы ещё на четырёх разбиениях. По этим прогнозам выбираем порог с наибольшим F1. При равенстве берём порог ближе к 0,5. После выбора порога обучаем модель на всей обучающей части и оцениваем её на внешнем фолде.

In [17]:
blocked_ids = listings.index[listings["close_type"] == "заблокировано"]

In [18]:
def choose_threshold(model, X_train, y_train, blocked_ids):
    inner_cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=INNER_SEED)
    train_probabilities = cross_val_predict(
        model, X_train, y_train, cv=inner_cv, method="predict_proba",
    )[:, 1]
    train_probabilities[X_train.index.isin(blocked_ids)] = 0

    return max(
        THRESHOLDS,
        key=lambda threshold: (
            f1_score(y_train, train_probabilities > threshold),
            -abs(threshold - 0.5),
        ),
    )

In [19]:
def evaluate_model(model, X, y, blocked_ids, seed):
    """Прогнозируем каждое объявление на фолде, где модель его не видела."""
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

    fold_metrics = []
    fold_predictions = []
    for fold, (train, test) in enumerate(cv.split(X, y), start=1):
        X_fit, y_fit = X.iloc[train], y.iloc[train]
        threshold = choose_threshold(model, X_fit, y_fit, blocked_ids)
        fold_model = clone(model).fit(X_fit, y_fit)

        y_test = y.iloc[test]
        p_deal = pd.Series(
            fold_model.predict_proba(X.iloc[test])[:, 1],
            index=y_test.index,
        )
        p_deal.loc[y_test.index.intersection(blocked_ids)] = 0
        labels = (p_deal > threshold).astype(int)

        fold_metrics.append({
            "Фолд": fold,
            "Порог": threshold,
            "f1": f1_score(y_test, labels),
            "precision": precision_score(y_test, labels),
            "recall": recall_score(y_test, labels),
            "roc_auc": roc_auc_score(y_test, p_deal),
            "accuracy": accuracy_score(y_test, labels),
        })
        fold_predictions.append(pd.DataFrame({
            "fold": fold,
            "is_deal_true": y_test,
            "is_deal_pred": labels,
            "p_deal": p_deal,
        }))

    oof_predictions = pd.concat(fold_predictions).loc[y.index]
    assert oof_predictions.index.equals(y.index), "Нужен один прогноз на каждое объявление"
    by_fold = pd.DataFrame(fold_metrics).set_index("Фолд")
    return oof_predictions, by_fold

def category_deal_shares(predictions, category):
    return predictions.join(category).groupby("category")[
        ["is_deal_true", "is_deal_pred"]
    ].mean()

In [20]:
oof_predictions, by_fold = evaluate_model(model, X_train, y, blocked_ids, seed=42)
metrics = by_fold.drop(columns="Порог")
summary = pd.DataFrame({
    "Среднее": metrics.mean(),
    "Стандартное отклонение": metrics.std(ddof=0),
})
print(f"Проверено объявлений: {len(oof_predictions)} из {len(verified)}")
print(f"Заблокированных в проверке: {oof_predictions.index.isin(blocked_ids).sum()}")
display(by_fold.round(4))
display(summary.round(4))

# Доли сделок в категориях по отложенным прогнозам.
category_shares = category_deal_shares(oof_predictions, features["category"])
cat_share_mae = (category_shares["is_deal_true"] - category_shares["is_deal_pred"]).abs().mean()

print("F1 по всем проверочным прогнозам:", round(
    f1_score(oof_predictions["is_deal_true"], oof_predictions["is_deal_pred"]), 4
))
print("CatShareMAE, п. п.:", round(100 * cat_share_mae, 2))
display(category_shares.round(4))

Проверено объявлений: 300 из 300
Заблокированных в проверке: 2


,Порог,f1,precision,recall,roc_auc,accuracy
Фолд,,,,,,
1,0.24,0.8148,0.7097,0.9565,0.9471,0.8333
2,0.34,0.8261,0.8261,0.8261,0.9424,0.8667
3,0.37,0.7083,0.6800,0.7391,0.9130,0.7667
4,0.44,0.6190,0.6842,0.5652,0.7767,0.7333
5,0.45,0.9167,0.9167,0.9167,0.9769,0.9333


,Среднее,Стандартное отклонение
f1,0.7770,0.1030
precision,0.7633,0.0933
recall,0.8007,0.1397
roc_auc,0.9112,0.0702
accuracy,0.8267,0.0712


F1 по всем проверочным прогнозам: 0.7815
CatShareMAE, п. п.: 3.42


,is_deal_true,is_deal_pred
category,,
Бытовая техника,0.4000,0.4667
Детские товары,0.4889,0.4889
Запчасти и аксессуары,0.3143,0.3714
Мебель и интерьер,0.4571,0.4857
Одежда и обувь,0.2698,0.2698
Спорт и отдых,0.2500,0.3214
Электроника,0.4844,0.4688


### Проверка новых признаков

Сравниваем базовую таблицу, каждый новый блок по отдельности и оба блока вместе. Разбиения и схема подбора порога одинаковы. Для итогового прогноза используем обе группы.

In [21]:
dialogue_columns = [
    "has_arrival_without_later_refusal",
    "has_mutual_meeting",
    "has_handoff_thanks",
    "buyer_refusal_share",
    "has_seller_sold",
]
buyer_contact_columns = [
    "buyers_with_chat_and_phone",
    "repeat_contact_buyer_share",
    "phone_viewers_last_24h",
]
feature_sets = {
    "Базовые признаки": X_train.drop(columns=dialogue_columns + buyer_contact_columns),
    "С этапами разговора": X_train.drop(columns=buyer_contact_columns),
    "С контактами покупателей": X_train.drop(columns=dialogue_columns),
    "С обоими блоками": X_train,
}

In [22]:
comparison_metrics = []
for seed in CV_SEEDS:
    for name, training_features in feature_sets.items():
        validation_predictions, _ = evaluate_model(model, training_features, y, blocked_ids, seed)
        shares = category_deal_shares(validation_predictions, features["category"])
        share_error = (shares["is_deal_true"] - shares["is_deal_pred"]).abs().mean()

        comparison_metrics.append({
            "Набор": name,
            "Разбиение": seed,
            "F1": f1_score(
                validation_predictions["is_deal_true"],
                validation_predictions["is_deal_pred"],
            ),
            "CatShareMAE, п. п.": 100 * share_error,
        })

comparison_results = pd.DataFrame(comparison_metrics)
display(comparison_results.round(4))
display(comparison_results.groupby("Набор", sort=False)[
    ["F1", "CatShareMAE, п. п."]
].mean().round(4))

,Набор,Разбиение,F1,"CatShareMAE, п. п."
0,Базовые признаки,11,0.7479,4.2297
1,С этапами разговора,11,0.7753,2.1995
2,С контактами покупателей,11,0.7561,5.1935
3,С обоими блоками,11,0.7792,1.5838
4,Базовые признаки,23,0.7458,2.9670
5,С этапами разговора,23,0.7949,2.2562
6,С контактами покупателей,23,0.7654,5.7979
7,С обоими блоками,23,0.8070,4.0781
8,Базовые признаки,42,0.7615,5.0234
9,С этапами разговора,42,0.7718,5.8014


,F1,"CatShareMAE, п. п."
Набор,,
Базовые признаки,0.7583,4.3229
С этапами разговора,0.7749,3.1276
С контактами покупателей,0.7704,4.5457
С обоими блоками,0.7924,3.0721


## 4. Обучить итоговую модель и получить прогнозы

Подбираем один порог по четырём фолдам на всей проверенной разметке. Затем обучаем модель на всех размеченных объявлениях и применяем этот порог к прогнозам по всему набору.

In [23]:
threshold = choose_threshold(model, X_train, y, blocked_ids)
print("Итоговый порог:", round(threshold, 2))

model.fit(X_train, y)
probabilities = model.predict_proba(X)[:, 1]
predictions = X.index.to_frame(index=False).assign(
    is_deal=(probabilities > threshold).astype(int),
)

Итоговый порог: 0.5


## 5. Сохранить CSV

Оставляем прогнозы модели для незаблокированных объявлений. Для заблокированных ставим `is_deal = 0`.

In [24]:
submission = predictions.copy()
submission.loc[submission["listing_id"].isin(blocked_ids), "is_deal"] = 0

assert submission.columns.tolist() == ["listing_id", "is_deal"]
assert submission["listing_id"].is_unique
assert set(submission["listing_id"]) == set(listings.index)
assert submission["is_deal"].isin([0, 1]).all()

submission.to_csv(OUTPUT_PATH, index=False)
print(f"Сохранено {len(submission)} объявлений в {OUTPUT_PATH}")
submission.head()

Сохранено 15274 объявлений в submission.csv


,listing_id,is_deal
0,132065,1
1,131440,0
2,117997,1
3,129810,1
4,111780,1
